# ZIPコード単位 Areal Deprivation Index（ADI）作成手順

## 1. 概要

本データは、2020年国勢調査の小地域データを用いて作成した **ZIPコード（郵便番号）単位の Areal Deprivation Index（ADI）** である。  
ADIは、社会経済的剥奪（socioeconomic deprivation）を表す合成指標であり、複数の人口・世帯・就業構成指標を加重合成して算出した。

本プロセスでは：

- 小地域（町丁・字レベル）でADIを作成
- 郵便番号ポリゴンに対して面積按分
- ZIP単位に集約

という2段階の空間統合を行っている。

---

## 2. 使用データ

### 2.1 国勢調査（2020年）　小地域集計

以下の統計表を使用：

| 表番号 | 内容 |
|------|------|
| 第6表 | 世帯構成 |
| 第7表 | 住宅の所有形態 |
| 第9表 | 労働力状態 |
| 第12表 | 職業 |

共通条件：

- 地域階層レベル：原則 Level 4（町丁・字）
- fallback：Level 4 が存在しない自治体のみ Level 3 を使用
- 秘匿地域：除外
- 男女：総数のみ
- 世帯分類：総数のみ（該当表）

---

### 2.2 小地域ポリゴン

- 出典：国勢調査2020 小地域ポリゴン（JGD2011）
- 座標系：世界測地系（緯度経度）
- 形式：Shapefile

---

### 2.3 郵便番号ポリゴン

- 出典：GEOSPACE 郵便番号ポリゴン（2022）
- 単位：ZIPコード（7桁）

---

## 3. 小地域レベルでのデータ作成

### 3.1 キーの統合

各表を以下のキーで結合：

```python
area_key = 市区町村コード + "_" + 町丁字コード
```



### 3.2 使用変数

以下の8指標を構成変数として使用：

| 変数名 | 定義 |
|--------|------|
| old_couple_rate | 高齢夫婦のみ世帯 / 全世帯 |
| old_single_rate | 高齢単身世帯 / 全世帯 |
| single_parent_proxy_rate | ひとり親世帯（proxy）/ 全世帯 |
| rent_rate | 借家世帯 / 住宅世帯 |
| sales_service_rate | 販売・サービス職比率 |
| agriculture_rate | 農林漁業従事者比率 |
| blue_collar_rate | ブルーカラー職比率 |
| unemployment_rate | 失業率 |

---

### 3.3 ADIの算出

以下の線形結合でADIを算出：

```python
ADI_raw =
  2.99 * old_couple_rate
+ 7.57 * old_single_rate
+ 17.4 * single_parent_proxy_rate
+ 2.22 * rent_rate
+ 4.03 * sales_service_rate
+ 6.05 * agriculture_rate
+ 5.38 * blue_collar_rate
+ 18.3 * unemployment_rate
```

---
## 4. 地域階層レベルの統合（重要）

### 4.1 fallbackルール

市区町村単位で：
	•	Level 4 が存在 → Level 4 を採用
	•	Level 4 が存在しない → Level 3 を採用

👉 同一自治体内で複数レベルは混在しない

---
### 4.2 目的
	•	山間部・離島などの欠損を補完
	•	空間カバレッジを最大化

---
## 5. 小地域ポリゴンとの結合

### 5.1 KEY_CODE の作成
```python
KEY_CODE = 市区町村コード（5桁） + 町丁字コード（6桁）
```

Level3/2 の場合は右ゼロ埋めで6桁化

---
### 5.2 ポリゴン処理
	•	KEY_CODE単位で dissolve
	•	geometry の整合性修正（make_valid）

---
## 6. 郵便番号への面積按分

### 6.1 空間結合
gpd.overlay(smallarea, zip, how="intersection")

---
### 6.2 重み計算
weight = 交差面積 / 小地域面積

---
### 6.3 品質確認
	•	小地域ごとの weight 合計 ≈ 1
	•	二重計上なし（max ≈ 1.000001）

---
## 7. ZIP単位への集約

### 7.1 加重平均

各変数について：
ZIP値 = Σ(値 × weight) / Σ(weight)
※ 欠損を除外した weight で分母を調整

---
### 7.2 ADIの扱い

本研究では：

👉 小地域で計算したADIを面積加重平均してZIPに集約

（※ 変数を先に集約してADIを再計算する方法とは異なる）

---
## 8. フィルタリング

3種類のデータを作成：

| データ | 条件 |
|--------|------|
| nofilter | ADIが存在するものすべて |
| th001 | weight_sum > 0.01 |
| th01 | weight_sum > 0.1 |

※ nofilter はカバレッジ最大、th01 は信頼性重視の分析向け

---

## 9. 分位カテゴリ

ZIP単位で以下のカテゴリ変数を作成：

- 5分位（ADI_q5）
- 4分位（ADI_q4）

例：

```python
zip_adi["ADI_q5"] = pd.qcut(zip_adi["ADI_raw"], q=5, labels=False)
zip_adi["ADI_q4"] = pd.qcut(zip_adi["ADI_raw"], q=4, labels=False)
```

## 10. 出力形式

本データは以下の形式で出力した：

| ファイル | 内容 |
|--------|------|
| parquet | 分析用（高速・軽量・推奨） |
| CSV | 確認・共有用 |
| GPKG | GIS可視化用 |

parquet形式を主とし、用途に応じて他形式を併用した。

---

## 11. 制約・注意点

### 11.1 カバレッジ

- 一部の郵便番号ポリゴンではADIが欠損となる  
- 主な理由は対応する小地域統計が存在しないため  
- 特に山間部・非居住地域で多く見られる  

---

### 11.2 空間的不一致

- 郵便番号ポリゴンと小地域ポリゴンは境界が一致しない  
- 本研究では面積按分により統合している  
- そのため境界付近では誤差が生じうる  

---

### 11.3 地域階層レベルの影響

- 一部自治体では最小単位がLevel4ではなくLevel3となる  
- fallbackによりLevel3を採用している  
- このため空間解像度に地域差が存在する  

---

### 11.4 proxy変数の限界

- single_parent_proxy_rate は厳密なひとり親世帯ではなく推定指標である  
- 解釈には注意が必要  

---

### 11.5 手法依存性

- 本研究では「小地域で算出したADIの加重平均」を採用  
- 「ZIP単位で構成変数を再計算してADIを作成」する方法とは一致しない可能性がある  

---

## 12. 再現性

本処理は以下の手順で再現可能：

1. 国勢調査小地域集計データの整形  
2. 小地域レベルでADI構成変数の作成  
3. ADIの算出  
4. 小地域ポリゴンとの結合  
5. overlay（intersection）による空間結合  
6. 面積に基づく重みの計算  
7. 郵便番号単位での加重平均  
8. フィルタおよび分位カテゴリの作成  

---

# ✨ まとめ

本研究で作成したADIは：

- 国勢調査小地域集計データを基盤とし  
- 小地域ポリゴン（JGD2011）と結合し  
- 面積按分により郵便番号単位へ統合した指標である  

この手法により、

- 高い空間カバレッジ  
- 異なる空間単位間の整合性  

を両立している。

また、nofilterおよび閾値別データを併用することで、  
カバレッジと推定精度のトレードオフを柔軟に扱うことが可能である。